# Project 1

Checking virtual environment

In [1]:
import sys

print(sys.executable)

/Users/tsampikakiaourtzi/projects/big_data_project/.venv311/bin/python


## Setting up Spark Environment

Find the right path

In [8]:
! /usr/libexec/java_home -V


Matching Java Virtual Machines (1):
    17.0.21.0.2 (arm64) "Homebrew" - "OpenJDK 17.0.21.0.2" /opt/homebrew/Cellar/openjdk@17/17.0.21.0.2/libexec/openjdk.jdk/Contents/Home
/opt/homebrew/Cellar/openjdk@17/17.0.21.0.2/libexec/openjdk.jdk/Contents/Home


Environmental variables

In [3]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.21.0.2/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [3]:
!pip install -q findspark

In [4]:
findspark.find()

NameError: name 'findspark' is not defined

In [5]:
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("Lab4Project1") \
    .master("local[*]") \
    .getOrCreate()  


26/10/26 21:08:42 WARN Utils: Your hostname, Tsampikas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.104 instead (on interface en0)
26/10/26 21:08:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 21:08:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
spark

## Data loading

In [7]:
df = spark.read.csv("../data/raw/star_classification.csv", header=True, inferSchema=True)
df.show(5)

+--------------------+----------------+------------------+--------+--------+--------+--------+--------+------+--------+-------+--------+--------------------+------+---------+-----+-----+--------+
|              obj_ID|           alpha|             delta|       u|       g|       r|       i|       z|run_ID|rerun_ID|cam_col|field_ID|         spec_obj_ID| class| redshift|plate|  MJD|fiber_ID|
+--------------------+----------------+------------------+--------+--------+--------+--------+--------+------+--------+-------+--------+--------------------+------+---------+-----+-----+--------+
|1.237660961327743...|  135.6891066036|  32.4946318397087|23.87882| 22.2753|20.39501|19.16573|18.79371|  3606|     301|      2|      79|6.543777369295181...|GALAXY|0.6347936| 5812|56354|     171|
|1.237664879951151...|144.826100550256|  31.2741848944939|24.77759|22.83188|22.58444|21.16812|21.61427|  4518|     301|      5|     119|1.176014203670733...|GALAXY| 0.779136|10445|58158|     427|
|1.237660961330430..

### Exploring

In [8]:
print("Rows: %d, Columns: %d" % (df.count(), len(df.columns)))  # check the number of rows and columns in the df
df.printSchema()

Rows: 100000, Columns: 18
root
 |-- obj_ID: double (nullable = true)
 |-- alpha: double (nullable = true)
 |-- delta: double (nullable = true)
 |-- u: double (nullable = true)
 |-- g: double (nullable = true)
 |-- r: double (nullable = true)
 |-- i: double (nullable = true)
 |-- z: double (nullable = true)
 |-- run_ID: integer (nullable = true)
 |-- rerun_ID: integer (nullable = true)
 |-- cam_col: integer (nullable = true)
 |-- field_ID: integer (nullable = true)
 |-- spec_obj_ID: double (nullable = true)
 |-- class: string (nullable = true)
 |-- redshift: double (nullable = true)
 |-- plate: integer (nullable = true)
 |-- MJD: integer (nullable = true)
 |-- fiber_ID: integer (nullable = true)



In [9]:
df.groupBy("class").count().show() # check the distribution of the classes

+------+-----+
| class|count|
+------+-----+
|GALAXY|59445|
|   QSO|18961|
|  STAR|21594|
+------+-----+



## Data preprocessing using SparkML

In [ ]:
### Select columns 

In [10]:
from pyspark.sql.functions import col

df = df.select(col("u"), col("g"), col("r"), col("i"), col("z"), col("redshift"), col("class"))
df.show(3)


+--------+--------+--------+--------+--------+---------+------+
|       u|       g|       r|       i|       z| redshift| class|
+--------+--------+--------+--------+--------+---------+------+
|23.87882| 22.2753|20.39501|19.16573|18.79371|0.6347936|GALAXY|
|24.77759|22.83188|22.58444|21.16812|21.61427| 0.779136|GALAXY|
|25.26307|22.66389|20.60976|19.34857|18.94827|0.6441945|GALAXY|
+--------+--------+--------+--------+--------+---------+------+
only showing top 3 rows



### Check null values

In [11]:
from pyspark.sql.functions import col, count, when, isnull

df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).show() # check for null values

+---+---+---+---+---+--------+-----+
|  u|  g|  r|  i|  z|redshift|class|
+---+---+---+---+---+--------+-----+
|  0|  0|  0|  0|  0|       0|    0|
+---+---+---+---+---+--------+-----+



### Convert the text column 'class' to numeric

In [12]:
from pyspark.ml.feature import StringIndexer

df = StringIndexer(inputCol="class", outputCol="class_index", handleInvalid="keep").fit(df).transform(df)

In [13]:
df.show(2)

+--------+--------+--------+--------+--------+---------+------+-----------+
|       u|       g|       r|       i|       z| redshift| class|class_index|
+--------+--------+--------+--------+--------+---------+------+-----------+
|23.87882| 22.2753|20.39501|19.16573|18.79371|0.6347936|GALAXY|        0.0|
|24.77759|22.83188|22.58444|21.16812|21.61427| 0.779136|GALAXY|        0.0|
+--------+--------+--------+--------+--------+---------+------+-----------+
only showing top 2 rows



In [17]:
df = df.drop("class")
df.show(5)

+--------+--------+--------+--------+--------+---------+-----------+
|       u|       g|       r|       i|       z| redshift|class_index|
+--------+--------+--------+--------+--------+---------+-----------+
|23.87882| 22.2753|20.39501|19.16573|18.79371|0.6347936|        0.0|
|24.77759|22.83188|22.58444|21.16812|21.61427| 0.779136|        0.0|
|25.26307|22.66389|20.60976|19.34857|18.94827|0.6441945|        0.0|
|22.13682|23.77656|21.61162|20.50454| 19.2501|0.9323456|        0.0|
|19.43718|17.58028|16.49747|15.97711|15.54461|0.1161227|        0.0|
+--------+--------+--------+--------+--------+---------+-----------+
only showing top 5 rows



### Combine all columns expect the label into a single vector column using VectorAssembler

In [18]:
df.printSchema()

root
 |-- u: double (nullable = true)
 |-- g: double (nullable = true)
 |-- r: double (nullable = true)
 |-- i: double (nullable = true)
 |-- z: double (nullable = true)
 |-- redshift: double (nullable = true)
 |-- class_index: double (nullable = false)



### Feature Engineering

In [19]:
from pyspark.ml.feature import VectorAssembler

features_columns = [col for col in df.columns if col != "class_index"]
assembler = VectorAssembler(inputCols=features_columns, outputCol="features")
transformed_df = assembler.transform(df)
transformed_df.show(5)



+--------+--------+--------+--------+--------+---------+-----------+--------------------+
|       u|       g|       r|       i|       z| redshift|class_index|            features|
+--------+--------+--------+--------+--------+---------+-----------+--------------------+
|23.87882| 22.2753|20.39501|19.16573|18.79371|0.6347936|        0.0|[23.87882,22.2753...|
|24.77759|22.83188|22.58444|21.16812|21.61427| 0.779136|        0.0|[24.77759,22.8318...|
|25.26307|22.66389|20.60976|19.34857|18.94827|0.6441945|        0.0|[25.26307,22.6638...|
|22.13682|23.77656|21.61162|20.50454| 19.2501|0.9323456|        0.0|[22.13682,23.7765...|
|19.43718|17.58028|16.49747|15.97711|15.54461|0.1161227|        0.0|[19.43718,17.5802...|
+--------+--------+--------+--------+--------+---------+-----------+--------------------+
only showing top 5 rows



## Model Training and Testing using SparkML

In [ ]:
# Split the data into training and testing sets
(training_data, testing_data) = transformed_df.randomSplit([0.8, 0.2], seed=42) # seed to ensure reproducibility
print("Training Data Count: %d" % training_data.count())
print("Testing Data Count: %d" % testing_data.count())

Training Data Count: 80175
Testing Data Count: 19825


In [22]:
# Create a Random Forest Classifier model
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="class_index", featuresCol="features", maxDepth=5, seed=42)
model = rf.fit(training_data)


## Prediction on Test data

In [24]:
predictions = model.transform(testing_data)
predictions.select("features", "class_index", "prediction").show(10)

+--------------------+-----------+----------+
|            features|class_index|prediction|
+--------------------+-----------+----------+
|[14.26381,12.0359...|        1.0|       1.0|
|[14.64972,12.6357...|        1.0|       0.0|
|[14.92536,13.5602...|        0.0|       0.0|
|[15.38325,14.3722...|        0.0|       0.0|
|[15.54097,13.7889...|        1.0|       1.0|
|[15.62139,13.7305...|        0.0|       0.0|
|[15.70966,13.8794...|        0.0|       0.0|
|[15.78794,14.8433...|        1.0|       1.0|
|[16.05739,14.8254...|        0.0|       0.0|
|[16.10737,14.4537...|        0.0|       0.0|
+--------------------+-----------+----------+
only showing top 10 rows



## Evaluation of predictions

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="class_index", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Test Accuracy = %g" % accuracy)  


Test Accuracy = 0.958789


In [ ]:
# Confusion Matrix: counts of true label  vs predicted label
predictions.groupBy("class_index", "prediction").count().orderBy("class_index", "prediction").show()

+-----------+----------+-----+
|class_index|prediction|count|
+-----------+----------+-----+
|        0.0|       0.0|11434|
|        0.0|       1.0|  181|
|        0.0|       2.0|  152|
|        1.0|       0.0|   14|
|        1.0|       1.0| 4286|
|        2.0|       0.0|  469|
|        2.0|       1.0|    1|
|        2.0|       2.0| 3288|
+-----------+----------+-----+



# Project 2

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

## Setting up the environment - Same as in Project 1

## Read data

In [27]:
df2 = spark.read.csv("../data/raw/star_classification.csv", header=True, inferSchema=True)

## Leverage Spark ML pipeline

In [35]:
# Select columns
df2 = df2.select(col("u"), col("g"), col("r"), col("i"), col("z"), col("redshift"), col("class"))

# Split the data into training and testing sets
training_data2, testing_data2 = df2.randomSplit([0.8, 0.2], seed=42)

# Label encoding for the "class" column
class_indexer = StringIndexer(inputCol="class", outputCol="class_index")

# Assemble features into a single vector column
inputCols = ["u", "g", "r", "i", "z", "redshift"]
outputCol = "features"
assembler2 = VectorAssembler(inputCols=inputCols, outputCol=outputCol) 

# Model training using Random Forest Classifier
rf2 = RandomForestClassifier(labelCol="class_index", featuresCol="features")

In [36]:
from pyspark.ml import Pipeline

# Set up the pipeline
pipeline = Pipeline(stages=[class_indexer, assembler2, rf2]) # transformers: class_indexer, assembler2 estimator: rf2

# Fit the pipeline model
final_pipeline = pipeline.fit(training_data)

# Predict on the testing data
predictions2 = final_pipeline.transform(testing_data)
predictions2.show(5)

+--------+--------+--------+--------+--------+------------+------+-----------+--------------------+--------------------+--------------------+----------+
|       u|       g|       r|       i|       z|    redshift| class|class_index|            features|       rawPrediction|         probability|prediction|
+--------+--------+--------+--------+--------+------------+------+-----------+--------------------+--------------------+--------------------+----------+
|14.26381|12.03599|11.64166|11.51527|12.55128| 1.579277E-4|  STAR|        1.0|[14.26381,12.0359...|[0.83891974202374...|[0.04194598710118...|       1.0|
|14.64972|12.63574| 12.1434|11.96598|13.13046| 2.895463E-4|  STAR|        1.0|[14.64972,12.6357...|[3.70230725554899...|[0.18511536277744...|       1.0|
|14.92536| 13.5602|13.10489|12.83076|12.64798| 0.006129465|GALAXY|        0.0|[14.92536,13.5602...|[3.70230725554899...|[0.18511536277744...|       1.0|
|15.38325|14.37224|13.87742|13.62063|13.30364|  0.01931902|GALAXY|        0.0|[15.

In [6]:
# Stop the SparkSession
spark.stop()